<a href="https://colab.research.google.com/github/khanRomaan/image-enhancement/blob/main/image_enhancement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install RAR extractor tool
!apt-get install unrar

# Extract the uploaded RAR file
!unrar x /content/Dataset.rar /content/


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/Dataset.rar

Creating    /content/Dataset                                          OK
Creating    /content/Dataset/HQ                                       OK
Creating    /content/Dataset/HQ/test                                  OK
Creating    /content/Dataset/HQ/test/images                           OK
Extracting  /content/Dataset/HQ/test/images/1.png                          0%  OK 
Extracting  /content/Dataset/HQ/test/images/10.png                         0%  OK 
Extracting  /content/Dataset/HQ/test/images/100.png                        0%  OK 
Extracting  /content/Dataset/HQ/test/images/101.png                        0%  OK 
Extract

In [ ]:
!find /content/Dataset -type f \( -iname "*.png" -o -iname "*.jpg" -o -iname "*.jpeg" \) | wc -l


1500


In [ ]:
import os
from PIL import Image, ImageFilter, ImageEnhance
import numpy as np

# Paths (adjust if needed)
hq_root = "/content/Dataset/HQ"
lq_root = "/content/Dataset/LQ"

# Degradation settings
blur_radius = 2
downsample_factor = 2
noise_std = 10
brightness_factor = 0.8
contrast_factor = 0.8

# List of splits
splits = ['train', 'val', 'test']

for split in splits:
    hq_split_folder = os.path.join(hq_root, split, "images")
    lq_split_folder = os.path.join(lq_root, split)
    os.makedirs(lq_split_folder, exist_ok=True)

    # Count images in HQ folder
    images = [f for f in os.listdir(hq_split_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    print(f"{split} → {len(images)} HQ images found.")

    for img_name in images:
        hq_path = os.path.join(hq_split_folder, img_name)
        img = Image.open(hq_path).convert('RGB')

        # Step 1: Gaussian blur
        lq_img = img.filter(ImageFilter.GaussianBlur(radius=blur_radius))

        # Step 2: Downsample and upscale
        w, h = lq_img.size
        lq_img = lq_img.resize((w // downsample_factor, h // downsample_factor))
        lq_img = lq_img.resize((w, h))

        # Step 3: Add Gaussian noise
        arr = np.array(lq_img).astype(np.float32)
        noise = np.random.normal(0, noise_std, arr.shape)
        arr += noise
        arr = np.clip(arr, 0, 255).astype(np.uint8)
        lq_img = Image.fromarray(arr)

        # Step 4: Adjust brightness and contrast
        enhancer = ImageEnhance.Brightness(lq_img)
        lq_img = enhancer.enhance(brightness_factor)
        enhancer = ImageEnhance.Contrast(lq_img)
        lq_img = enhancer.enhance(contrast_factor)

        # Save LQ image
        lq_path = os.path.join(lq_split_folder, img_name)
        lq_img.save(lq_path)

    print(f"{split} LQ images created in {lq_split_folder}\n")

print("All LQ images created successfully!")


train → 1000 HQ images found.
train LQ images created in /content/Dataset/LQ/train

val → 200 HQ images found.
val LQ images created in /content/Dataset/LQ/val

test → 300 HQ images found.
test LQ images created in /content/Dataset/LQ/test

All LQ images created successfully!


In [ ]:
import shutil

# Compress the Dataset folder
shutil.make_archive('/content/Dataset', 'zip', '/content/Dataset')

# This will create /content/Dataset.zip


KeyboardInterrupt: 

In [ ]:
import os

# Paths
hq_root = "/content/Dataset/HQ"
lq_root = "/content/Dataset/LQ"

splits = ['train', 'val', 'test']

print("=== HQ Images Count Per Split ===")
for split in splits:
    hq_folder = os.path.join(hq_root, split, "images")
    if not os.path.exists(hq_folder):
        print(f"{split} → Folder not found!")
        continue
    hq_images = [f for f in os.listdir(hq_folder) if f.lower().endswith(('.png','.jpg','.jpeg'))]
    print(f"{split} → {len(hq_images)} images")

print("\n=== Empty LQ Subfolders Check ===")
for split in splits:
    lq_folder = os.path.join(lq_root, split)
    if os.path.exists(lq_folder):
        files = os.listdir(lq_folder)
        if len(files) == 0:
            print(f"Empty: {lq_folder}")
    else:
        print(f"Missing LQ folder: {lq_folder}")

print("\n=== HQ vs LQ Filename Match Check ===")
for split in splits:
    hq_folder = os.path.join(hq_root, split, "images")
    lq_folder = os.path.join(lq_root, split)

    hq_images = sorted([f for f in os.listdir(hq_folder) if f.lower().endswith(('.png','.jpg','.jpeg'))])
    lq_images = sorted([f for f in os.listdir(lq_folder) if f.lower().endswith(('.png','.jpg','.jpeg'))])

    match = hq_images == lq_images
    print(f"{split} Match: {match} (HQ: {len(hq_images)} | LQ: {len(lq_images)})")


=== HQ Images Count Per Split ===
train → 1000 images
val → 200 images
test → 300 images

=== Empty LQ Subfolders Check ===

=== HQ vs LQ Filename Match Check ===
train Match: True (HQ: 1000 | LQ: 1000)
val Match: True (HQ: 200 | LQ: 200)
test Match: True (HQ: 300 | LQ: 300)


In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from einops import rearrange
from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio, structural_similarity


In [ ]:
# Device-agnostic (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [ ]:
class AngioDataset(Dataset):
    def __init__(self, lq_dir, hq_dir, image_size=256):
        self.lq_paths = sorted([
            os.path.join(lq_dir, f)
            for f in os.listdir(lq_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        self.hq_paths = sorted([
            os.path.join(hq_dir, f)
            for f in os.listdir(hq_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])

        assert len(self.lq_paths) == len(self.hq_paths), \
            "Mismatch between LQ and HQ images"

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((image_size, image_size))
        ])

    def __len__(self):
        return len(self.lq_paths)

    def __getitem__(self, idx):
        lq = cv2.imread(self.lq_paths[idx], cv2.IMREAD_GRAYSCALE)
        hq = cv2.imread(self.hq_paths[idx], cv2.IMREAD_GRAYSCALE)

        if lq is None or hq is None:
            raise RuntimeError("Image not found")

        lq = self.transform(lq)
        hq = self.transform(hq)

        return lq, hq


In [ ]:
class AngioDataset(Dataset):
    def __init__(self, lq_dir, hq_dir, image_size=256):
        self.lq_paths = sorted([
            os.path.join(lq_dir, f)
            for f in os.listdir(lq_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        self.hq_paths = sorted([
            os.path.join(hq_dir, f)
            for f in os.listdir(hq_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])

        assert len(self.lq_paths) == len(self.hq_paths), \
            "Mismatch between LQ and HQ images"

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((image_size, image_size))
        ])

    def __len__(self):
        return len(self.lq_paths)

    def __getitem__(self, idx):
        lq = cv2.imread(self.lq_paths[idx], cv2.IMREAD_GRAYSCALE)
        hq = cv2.imread(self.hq_paths[idx], cv2.IMREAD_GRAYSCALE)

        if lq is None or hq is None:
            raise RuntimeError("Image not found")

        lq = self.transform(lq)
        hq = self.transform(hq)

        return lq, hq


In [ ]:
class CNNEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
class CNNEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim=256, heads=2):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        b, c, h, w = x.shape
        x = rearrange(x, 'b c h w -> b (h w) c')
        x, _ = self.attn(x, x, x)
        x = self.norm(x)
        x = rearrange(x, 'b (h w) c -> b c h w', h=h)
        return x


In [ ]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 1, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
class HybridEnhancer(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = CNNEncoder()
        self.transformer = TransformerBlock()
        self.decoder = Decoder()

    def forward(self, x):
        x = self.encoder(x)
        x = self.transformer(x)
        x = self.decoder(x)
        return x


In [ ]:
train_dataset = AngioDataset(
    lq_dir="Dataset/LQ/train",
    hq_dir="Dataset/HQ/train/images",
        image_size=128
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True
)


In [ ]:
def train_model(model, dataloader, epochs=5):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()
    loss_history = []

    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for lq, hq in tqdm(dataloader):
            lq = lq.to(device)
            hq = hq.to(device)

            optimizer.zero_grad()
            output = model(lq)
            loss = criterion(output, hq)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(dataloader)
        loss_history.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

    return loss_history


In [ ]:
import torch
torch.cuda.empty_cache()


In [ ]:
model = HybridEnhancer().to(device)
losses = train_model(model, train_loader)


100%|██████████| 1000/1000 [06:06<00:00,  2.73it/s]


Epoch 1/5 - Loss: 0.0145


100%|██████████| 1000/1000 [06:14<00:00,  2.67it/s]


Epoch 2/5 - Loss: 0.0028


100%|██████████| 1000/1000 [06:14<00:00,  2.67it/s]


Epoch 3/5 - Loss: 0.0008


100%|██████████| 1000/1000 [06:13<00:00,  2.68it/s]


Epoch 4/5 - Loss: 0.0004


100%|██████████| 1000/1000 [06:13<00:00,  2.68it/s]

Epoch 5/5 - Loss: 0.0004


In [ ]:
def evaluate(model, dataloader):
    model.eval()
    psnr_list, ssim_list = [], []

    with torch.no_grad():
        for lq, hq in dataloader:
            lq = lq.to(device)
            hq = hq.to(device)

            output = model(lq)

            out = output.cpu().numpy()[0,0]
            gt = hq.cpu().numpy()[0,0]

            psnr_list.append(peak_signal_noise_ratio(gt, out))
            ssim_list.append(structural_similarity(gt, out))

    print("Average PSNR:", np.mean(psnr_list))
    print("Average SSIM:", np.mean(ssim_list))


In [ ]:
from skimage.metrics import structural_similarity, peak_signal_noise_ratio




In [ ]:
from skimage.metrics import structural_similarity, peak_signal_noise_ratio
import numpy as np

def evaluate(model, loader):
    model.eval()
    psnr_vals, ssim_vals = [], []

    with torch.no_grad():
        for lq, hq in loader:
            lq = lq.to(device)
            hq = hq.to(device)

            pred = model(lq)
            pred = torch.clamp(pred, 0, 1)

            pred_np = pred.squeeze().cpu().numpy().astype(np.float32)
            hq_np = hq.squeeze().cpu().numpy().astype(np.float32)

            if pred_np.ndim == 3:
                pred_np = pred_np[0]
                hq_np = hq_np[0]

            psnr_vals.append(
                peak_signal_noise_ratio(hq_np, pred_np, data_range=1.0)
            )

            ssim_vals.append(
                structural_similarity(hq_np, pred_np, data_range=1.0)
            )

    return float(np.mean(psnr_vals)), float(np.mean(ssim_vals))


In [ ]:
psnr, ssim = evaluate(model, train_loader)
print(f"PSNR: {psnr:.2f} dB")
print(f"SSIM: {ssim:.4f}")



PSNR: 36.12 dB
SSIM: 0.9682


In [ ]:
val_dataset = AngioDataset(
    lq_dir="/content/Dataset/LQ/val",
    hq_dir="/content/Dataset/HQ/val/images",
    image_size=128
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False
)


In [ ]:
import torch.nn as nn

class CNNEnhancer(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 1, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
cnn_model = CNNEnhancer().to(device)


In [ ]:
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-4)
criterion = nn.L1Loss()

epochs = 5

for epoch in range(epochs):
    cnn_model.train()
    total_loss = 0

    for lq, hq in train_loader:
        lq, hq = lq.to(device), hq.to(device)

        pred = cnn_model(lq)
        loss = criterion(pred, hq)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[CNN] Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")


[CNN] Epoch 1/5 | Loss: 0.0395
[CNN] Epoch 2/5 | Loss: 0.0221
[CNN] Epoch 3/5 | Loss: 0.0203
[CNN] Epoch 4/5 | Loss: 0.0190
[CNN] Epoch 5/5 | Loss: 0.0187


In [ ]:
psnr_cnn, ssim_cnn = evaluate(cnn_model, val_loader)
psnr_hyb, ssim_hyb = evaluate(model, val_loader)

print(f"CNN-only  → PSNR: {psnr_cnn:.2f}, SSIM: {ssim_cnn:.4f}")
print(f"Hybrid    → PSNR: {psnr_hyb:.2f}, SSIM: {ssim_hyb:.4f}")


CNN-only  → PSNR: 33.69, SSIM: 0.9594
Hybrid    → PSNR: 36.10, SSIM: 0.9670


In [ ]:
import torchvision.utils as vutils
import os

os.makedirs("results", exist_ok=True)

lq, hq = next(iter(val_loader))
lq = lq.to(device)

with torch.no_grad():
    pred = model(lq)

vutils.save_image(lq, "results/LQ.png")
vutils.save_image(pred, "results/Enhanced.png")
vutils.save_image(hq, "results/HQ.png")
